# Experiment 4A — نسخه عیب‌یابی مرحله Explore

این نسخه فقط **گزارش خطا و ثبت لاگ** را به نوت‌بوک اضافه می‌کند؛ منطق آزمایش و فایل `experiment4A_physical_value.py` تغییر نکرده.

سلول‌ها را از ابتدا اجرا کن. اگر `explore` خطا داد، علت واقعی از برنامه زیرمجموعه در خروجی و داخل فایل `experiment4a_results/explore_debug.log` ذخیره می‌شود. خروجی‌های قبلی حذف نمی‌شوند.


In [1]:
from pathlib import Path
import sys, os, subprocess, pandas as pd

EXP4A_DIR = Path.cwd()
PROJECT_ROOT = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm")
BENCH_DIR = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks")
MUTATIONS = EXP4A_DIR / "data" / "adversarial_mutation_cases.csv"
AUDIT = EXP4A_DIR / "data" / "width_and_original_plan_audit.csv"
PLANS = PROJECT_ROOT / "experiment2_results" / "experiment2_plan_features.csv"
SCRIPT = EXP4A_DIR / "experiment4A_physical_value.py"
OUT = EXP4A_DIR / "experiment4a_results"

print("Python:", sys.executable)
for desc, p in [("Experiment script", SCRIPT), ("Full Coverage CSV", MUTATIONS),
                ("Width audit", AUDIT), ("Experiment 2 plans", PLANS),
                ("QASM folder", BENCH_DIR), ("Project root", PROJECT_ROOT)]:
    print(f"{desc}: {p.exists()} — {p}")
if not SCRIPT.is_file():
    raise FileNotFoundError("فایل اسکریپت پیدا نشد؛ نوت‌بوک را در فولدر Experiment4A_Ready_Package اجرا کن.")


Python: C:\Users\ASUS\caqr-qiskit045-env\Scripts\python.exe
Experiment script: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\experiment4A_physical_value.py
Full Coverage CSV: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\adversarial_mutation_cases.csv
Width audit: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\width_and_original_plan_audit.csv
Experiment 2 plans: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment2_results\experiment2_plan_features.csv
QASM folder: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks
Project root: True — C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm


In [2]:
def run_stage(stage, max_new_routes=None, retry_errors=False):
    """Stream subprocess output AND save diagnostics; do not hide the real exception."""
    cmd = [sys.executable, str(SCRIPT), stage,
           "--mutations", str(MUTATIONS), "--audit", str(AUDIT),
           "--out", str(OUT)]
    if stage in ("explore", "validate"):
        if not PLANS.is_file() or not BENCH_DIR.is_dir():
            raise FileNotFoundError("PLANS یا BENCH_DIR اشتباه است؛ مسیرها را در سلول قبلی بررسی کن.")
        cmd += ["--plans", str(PLANS), "--bench-dir", str(BENCH_DIR)]
    if max_new_routes is not None:
        cmd += ["--max-new-routes", str(max_new_routes)]
    if retry_errors:
        cmd += ["--retry-errors"]
    print("Running:", " ".join(map(str, cmd)), flush=True)
    OUT.mkdir(parents=True, exist_ok=True)
    log_path = OUT / f"{stage}_debug.log"
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    last_lines = []
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(cmd, cwd=str(PROJECT_ROOT) if PROJECT_ROOT.is_dir() else str(EXP4A_DIR),
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace",
                              bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                last_lines.append(line.rstrip())
                if len(last_lines) > 35:
                    last_lines.pop(0)
            code = process.wait()
    if code:
        print("\n----- LAST 35 LINES OF REAL ERROR -----")
        print("\n".join(last_lines))
        print("---------------------------------------")
        print("Full error log:", log_path)
        show_diagnostics()
        raise RuntimeError(f"{stage} failed with exit code {code}; see the REAL error above / log file")
    print(f"{stage} succeeded. Log: {log_path}")
    return code

def show_diagnostics():
    """Summarize saved routing errors without rerunning expensive routes."""
    for name in ("routes_train.csv", "routes_holdout.csv"):
        path = OUT / name
        if not path.exists():
            print(f"No {name} yet (probably failed before routing began).")
            continue
        try:
            df = pd.read_csv(path)
            print(f"{name}: {len(df)} saved routing attempts")
            if "status" in df:
                print("Statuses:", df.status.value_counts(dropna=False).to_dict())
                failed = df[df.status.astype(str) != "ok"]
                if len(failed):
                    cols = [x for x in ("circuit", "candidate_key", "seed", "status", "error") if x in df]
                    print("First failing routes:")
                    print(failed[cols].head(8).to_string(index=False))
        except Exception as e:
            print(f"Could not read {name}: {e}")


## مرحلهٔ ۱ — آماده‌سازی بدون Routing

این مرحله تعداد همسایه‌ها را بررسی می‌کند و مجموعه کاندیداهای متمایز را می‌سازد.

In [3]:
run_stage('prepare')
import pandas as pd
print(pd.read_csv(OUT / 'coverage_per_source.csv').select_dtypes(include='number').sum())

Running: C:\Users\ASUS\caqr-qiskit045-env\Scripts\python.exe C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\experiment4A_physical_value.py prepare --mutations C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\adversarial_mutation_cases.csv --audit C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\width_and_original_plan_audit.csv --out C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\experiment4a_results
Input audit: {
  "input_mutations": 8642,
  "statically_valid_mutations": 485,
  "sources": 50,
  "circuits": 5,
  "all_distinct_physical_plans_including_originals_if_supplied": 135,
  "unique_valid_neighborhoods_per_source_total": 291,
  "unique_suffix_per_source_total": 130,
  "extra_from_chain_total": 159,
  "extra_from_other_total": 2,
  "originals_inc

## مرحله ۲ — Explore / گرفتن خطای واقعی

ابتدا بهتر است با دستور کوتاه `run_stage("explore", max_new_routes=1)` تست کنی. بعد برای اجرای کامل `run_stage("explore")` را اجرا کن. اگر قبلاً خروجی ذخیره شده باشد، اجرا از Checkpoint ادامه پیدا می‌کند.

این نوت‌بوک خودِ خطای Python و جزئیات شکست Transpile را به‌جای فقط `exit code 1` نمایش می‌دهد.


In [4]:
# اول وضعیت نتایج ذخیره‌شده از اجرای قبلی را ببین (بدون اجرای Routing جدید):
show_diagnostics()


No routes_train.csv yet (probably failed before routing began).
No routes_holdout.csv yet (probably failed before routing began).


In [5]:
# برای تست کوتاه، خط زیر را از حالت کامنت خارج کن:
# run_stage("explore", max_new_routes=1)

# اجرای واقعی (اگر خطا بدهد، جزئیات واقعی بالا نمایش داده می‌شود):
run_stage("explore")

if (OUT / "opportunity_summary.csv").is_file():
    display(pd.read_csv(OUT / "opportunity_summary.csv"))


Running: C:\Users\ASUS\caqr-qiskit045-env\Scripts\python.exe C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\experiment4A_physical_value.py explore --mutations C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\adversarial_mutation_cases.csv --audit C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\data\width_and_original_plan_audit.csv --out C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\Experiment4A_Ready_Package\experiment4a_results --plans C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment2_results\experiment2_plan_features.csv --bench-dir C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks
Input audit: {
  "input_mutations": 8642,
  "statically_valid_mutations": 485,
  "sources": 50,
  "circuits": 5,
  "all_distinct_p

RuntimeError: explore failed with exit code 1; see the REAL error above / log file

## مرحلهٔ ۳ — ارزیابی Frozen Winners با Seedهای تازهٔ ۱۰۰ تا ۱۲۹

انتخاب‌ها از مرحلهٔ قبل فریز شده‌اند. این مرحله برای جلوگیری از انتخاب دوباره بر اساس نتیجهٔ تصادفی Routing است.

In [ ]:
run_stage("validate")
if (OUT / "fresh_seed_comparison.csv").is_file():
    display(pd.read_csv(OUT / "fresh_seed_comparison.csv").head(15))


## تفسیر درست نتایج

اگر `strict_beneficiaries` مثبت بود، همسایگی گسترده برای آن معیار و آن Plan اولیه **فرصت جدید** ساخته است. اما این آزمایش نشان نمی‌دهد که یک الگوریتم با بودجهٔ محدود بتواند به آن برسد؛ آن مسئلهٔ آزمایش بعدی (Budgeted Guided Repair) است.

این فایل اعتبار ایستای Check شده را مبنا قرار می‌دهد؛ Semantics اجرایی Measurement/Reset را مستقلاً اثبات نمی‌کند.